# MSigDB GPU Hypergeometric ORA Saturation Analysis (study subsampling)

**Environment:** `gpu-kmeans` (Python kernel, cupy/RAPIDS)

For each CLAMP model in the study-subsampling saturation grid (varying K across coverage
levels, up to 3 seeds), this notebook:

1. Discovers the on-disk (rs_pct, K, seed) grid dynamically (no hardcoded/capped K list --
   the full grid up to K=1728 already exists on disk; only the Python `blitzgsea_gsea`
   notebooks artificially capped K to {86,173,432} due to CPU runtime, which doesn't apply
   here).
2. Adds a shared rs_pct=100 anchor (K=1728, all 3 seeds) reused from the random-subsampling
   coverage track's full-coverage model (`output/01_model_building/04_archs4/06_bp_coverage_rshall/06_bp_coverage_hall_rs_100`), since no
   saturation track has its own native 100%-coverage point -- mirrors the sibling
   `blitzgsea_gsea` notebooks' shared-anchor convention, but with all 3 seeds instead of 1.
3. Loads the Z matrix (gene loadings per LV) for each (rs_pct, K, seed) combination.
4. Gene universe = all model genes (`Z.index`), matching `clusterProfiler::enricher()`'s
   convention.
5. Runs a one-sided hypergeometric ORA test for every (LV, pathway) pair simultaneously via
   a GPU-vectorized kernel (`libs/gpu_ora.py`): hit set = top 1% of genes per LV, pathway DB =
   MSigDB v2026.1 filtered to size [10, 50000] within the universe.
6. BH-adjusts p-values within each LV, then takes the minimum adjusted p-value per pathway
   across all LVs.
7. Saves per-model CSV caches (`rs{rs_pct}_k{k_val}_seed{seed}_msigdb_gpu_ora.csv` +
   sibling `_meta.csv`) and per-rs_pct-level summary CSVs.

Missing (rs_pct, K, seed) combinations on disk (upstream model-building gaps, not a bug here)
are skipped gracefully.


In [1]:
import os
import re
import time

import pandas as pd

REPO_ROOT = "/home/msubirana/Documents/pivlab/clamp-analyses"
os.chdir(REPO_ROOT)

import sys
sys.path.insert(0, os.path.join(REPO_ROOT, "libs"))
import gpu_ora

t_start = time.time()

## Paths

In [2]:
models_dir = "output/01_model_building/04_archs4/08_saturation_study"
output_dir = "output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_msigdb"

os.makedirs(os.path.join(output_dir, "CLAMPfull"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "CLAMPbase"), exist_ok=True)

## Model grid: discovered dynamically from disk

Scans `08_saturation_study` for all `study_saturation_rs{pct}_k{k}_seed_{seed}` directories
that have a `CLAMPfull_hall/Z.csv`. K values and rs_pct levels are not hardcoded.

In [3]:
pattern = re.compile(r"^study_saturation_rs(\d+)_k(\d+)_seed_(\d+)$")

rows = []
for d in sorted(os.listdir(models_dir)):
    m = pattern.match(d)
    if not m:
        continue
    rs_pct, k_val, seed = int(m.group(1)), int(m.group(2)), int(m.group(3))
    z_path_full = os.path.join(models_dir, d, "CLAMPfull_hall", "Z.csv")
    if not os.path.exists(z_path_full):
        continue
    rows.append({
        "rs_pct": rs_pct, "k_val": k_val, "seed": seed, "subdir": d,
        "z_path_full": z_path_full,
        "z_path_base": os.path.join(models_dir, d, "CLAMPbase", "Z.csv"),
    })

model_grid = pd.DataFrame(rows).sort_values(["rs_pct", "k_val", "seed"]).reset_index(drop=True)
print(f"Available models: {len(model_grid)}")
print(model_grid[["rs_pct", "k_val", "seed"]])

Available models: 71
    rs_pct  k_val  seed
0        1     86     1
1        1     86     2
2        1     86     3
3        1    173     1
4        1    173     2
..     ...    ...   ...
66      25   1296     2
67      25   1296     3
68      25   1728     1
69      25   1728     2
70      25   1728     3

[71 rows x 3 columns]


## Shared rs_pct=100 anchor (K=1728, all 3 seeds)

No saturation track has its own native 100%-coverage model, so this reuses the
random-subsampling coverage track's full-coverage model (same physical model shared by
`coverage_study`/`saturation_random`/`saturation_study`).

In [4]:
anchor_models_dir = "output/01_model_building/04_archs4/06_bp_coverage_rshall"
anchor_dir = "06_bp_coverage_hall_rs_100"
anchor_prefix = "hall_coverage"

anchor_rows = []
for seed in [1, 2, 3]:
    seed_dir = os.path.join(anchor_models_dir, anchor_dir, f"{anchor_prefix}_rs100_seed_{seed}")
    z_path_full = os.path.join(seed_dir, "CLAMPfull_hall", "Z.csv")
    if not os.path.exists(z_path_full):
        print(f"SKIP anchor seed{seed} (no Z.csv): {z_path_full}")
        continue
    anchor_rows.append({
        "rs_pct": 100, "k_val": 1728, "seed": seed, "subdir": f"anchor_seed_{seed}",
        "z_path_full": z_path_full,
        "z_path_base": os.path.join(seed_dir, "CLAMPbase", "Z.csv"),
    })

model_grid = pd.concat([model_grid, pd.DataFrame(anchor_rows)], ignore_index=True)
model_grid = model_grid.sort_values(["rs_pct", "k_val", "seed"]).reset_index(drop=True)
print(f"Available models (incl. 100% anchor): {len(model_grid)}")

Available models (incl. 100% anchor): 74


## Load MSigDB gene sets

In [5]:
library = gpu_ora.read_gmt("data/pathways/msigdb.v2026.1.Hs.symbols.gmt")
print(f"MSigDB gene sets loaded: {len(library)}")

MSigDB gene sets loaded: 35361


## Helper: run GPU ORA across the full discovered grid for one model type

In [6]:
def run_model_grid(model_subdir, z_col):
    summary_rows = []

    for _, spec in model_grid.iterrows():
        rs_pct, k_val, seed = int(spec["rs_pct"]), int(spec["k_val"]), int(spec["seed"])
        z_path = spec[z_col]

        if not os.path.exists(z_path):
            print(f"SKIP (no Z.csv): {model_subdir} rs{rs_pct} k{k_val} seed{seed}")
            continue

        cache_path = os.path.join(output_dir, model_subdir, f"rs{rs_pct}_k{k_val}_seed{seed}_msigdb_gpu_ora.csv")
        meta_path = os.path.join(output_dir, model_subdir, f"rs{rs_pct}_k{k_val}_seed{seed}_meta.csv")

        if os.path.exists(cache_path) and os.path.exists(meta_path):
            print(f"Loading cached: {model_subdir} rs{rs_pct} k{k_val} seed{seed}")
            meta = pd.read_csv(meta_path).iloc[0]
        else:
            print(f"Running GPU ORA: {model_subdir} rs{rs_pct} k{k_val} seed{seed}")
            t0 = time.time()
            res = gpu_ora.run_gpu_ora_for_model(z_path, library, min_size=10, max_size=50000, pct=0.01)
            print(f"  done in {time.time()-t0:.2f}s")
            res["terms_padj"].rename_axis("term").reset_index(name="padj_min_across_lvs").to_csv(cache_path, index=False)
            meta = pd.Series({k: v for k, v in res.items() if k != "terms_padj"})
            meta.to_frame().T.to_csv(meta_path, index=False)

        summary_rows.append({
            "model_type": model_subdir, "rs_pct": rs_pct, "k_val": k_val, "seed": seed,
            "n_samples": meta["n_samples"], "n_lvs": meta["n_lvs"],
            "n_top_genes": meta["n_top_genes"], "n_total_msigdb": meta["n_total_msigdb"],
        })

    summary_df = pd.DataFrame(summary_rows).sort_values(["rs_pct", "k_val", "seed"]).reset_index(drop=True)

    for rs_pct in sorted(summary_df["rs_pct"].unique()):
        pct_df = summary_df[summary_df["rs_pct"] == rs_pct]
        pct_csv_path = os.path.join(output_dir, model_subdir, f"results_pct{rs_pct}_msigdb_gpu_ora.csv")
        pct_df.to_csv(pct_csv_path, index=False)
        print(f"Saved {model_subdir} rs{rs_pct}: {len(pct_df)} models -> {pct_csv_path}")

    return summary_df

## Run GPU ORA: CLAMPfull

In [7]:
results_full_df = run_model_grid("CLAMPfull", "z_path_full")
print(f"Collected {len(results_full_df)} CLAMPfull rows")
print(results_full_df)

Loading cached: CLAMPfull rs1 k86 seed1
Loading cached: CLAMPfull rs1 k86 seed2
Loading cached: CLAMPfull rs1 k86 seed3
Loading cached: CLAMPfull rs1 k173 seed1
Loading cached: CLAMPfull rs1 k173 seed2
Loading cached: CLAMPfull rs1 k173 seed3
Loading cached: CLAMPfull rs1 k432 seed1
Loading cached: CLAMPfull rs1 k432 seed2
Loading cached: CLAMPfull rs1 k432 seed3
Loading cached: CLAMPfull rs1 k864 seed1
Loading cached: CLAMPfull rs1 k864 seed2
Loading cached: CLAMPfull rs1 k864 seed3
Loading cached: CLAMPfull rs1 k1296 seed1
Loading cached: CLAMPfull rs1 k1296 seed2
Loading cached: CLAMPfull rs1 k1296 seed3
Loading cached: CLAMPfull rs1 k1728 seed2
Loading cached: CLAMPfull rs1 k1728 seed3
Loading cached: CLAMPfull rs5 k86 seed1
Loading cached: CLAMPfull rs5 k86 seed2
Loading cached: CLAMPfull rs5 k86 seed3
Loading cached: CLAMPfull rs5 k173 seed1
Loading cached: CLAMPfull rs5 k173 seed2
Loading cached: CLAMPfull rs5 k173 seed3
Loading cached: CLAMPfull rs5 k432 seed1
Loading cached: C

## Run GPU ORA: CLAMPbase

In [8]:
results_base_df = run_model_grid("CLAMPbase", "z_path_base")
print(f"Collected {len(results_base_df)} CLAMPbase rows")
print(results_base_df)

Loading cached: CLAMPbase rs1 k86 seed1
Loading cached: CLAMPbase rs1 k86 seed2
Loading cached: CLAMPbase rs1 k86 seed3
Loading cached: CLAMPbase rs1 k173 seed1
Loading cached: CLAMPbase rs1 k173 seed2
Loading cached: CLAMPbase rs1 k173 seed3
Loading cached: CLAMPbase rs1 k432 seed1
Loading cached: CLAMPbase rs1 k432 seed2
Loading cached: CLAMPbase rs1 k432 seed3
Loading cached: CLAMPbase rs1 k864 seed1
Loading cached: CLAMPbase rs1 k864 seed2
Loading cached: CLAMPbase rs1 k864 seed3
Loading cached: CLAMPbase rs1 k1296 seed1
Loading cached: CLAMPbase rs1 k1296 seed2
Loading cached: CLAMPbase rs1 k1296 seed3
Loading cached: CLAMPbase rs1 k1728 seed2
Loading cached: CLAMPbase rs1 k1728 seed3
Loading cached: CLAMPbase rs5 k86 seed1
Loading cached: CLAMPbase rs5 k86 seed2
Loading cached: CLAMPbase rs5 k86 seed3
Loading cached: CLAMPbase rs5 k173 seed1
Loading cached: CLAMPbase rs5 k173 seed2
Loading cached: CLAMPbase rs5 k173 seed3
Loading cached: CLAMPbase rs5 k432 seed1
Loading cached: C

In [9]:
print(f"Total notebook time: {(time.time()-t_start)/60:.1f} min")

Total notebook time: 0.0 min
